# song-lingo 연습 화면 (practice)

시연 화면입니다. 위에서 아래로 실행합니다. 구현된 모듈이 늘어날 때마다 셀을 추가합니다.

- 녹음은 노트북에서 직접 하지 않고 **미리 녹음한 파일 경로**를 지정합니다.
- 이 노트북의 출력에는 목소리·경로·key가 남을 수 있으므로 **커밋 전에 출력을 지우고** `scripts/scan_outputs.py`로 확인합니다.

In [ ]:
import sys
print(sys.executable)   # .../envs/voice-ai/bin/python 이어야 합니다

from dotenv import load_dotenv
load_dotenv()            # .env 의 OPENAI_API_KEY 를 읽습니다 (값은 출력하지 않습니다)

import scoring

## 0. 채점 동작 확인 (API 호출 없음)

In [ ]:
ref = "The cat sat on the mat."
hyp = "the cat sit on mat"

r = scoring.wer(ref, hyp)
print(f"WER {r['rate']:.3f}  (치환 {r['S']}, 삭제 {r['D']}, 삽입 {r['I']} / 정답 {r['N']}단어)")
print(scoring.format_diff(r['ops']))

## S0. 준비 확인 (스켈레톤)

키가 읽히는지(값은 출력하지 않음), Whisper `tiny` 모델이 로딩되는지 봅니다. 모델은 학습 저장소의 `models/`를 복사해 재사용합니다.

In [ ]:
import os
print("키 존재:", bool(os.getenv("OPENAI_API_KEY")))
print("커널 python:", sys.executable)

In [ ]:
import time
import stt   # stt.py: 녹음 찾기와 faster-whisper 인식

t0 = time.time()
model = stt.load_model()   # tiny + int8, 처음 한 번만 읽어 stt 모듈에 보관합니다
print(f"모델 로딩 성공 ({time.time() - t0:.1f}초)")

## S1·S2. 연습 문장과 기준 음성 (TTS)

`sentences.csv`에서 문장을 읽고, 1번 문장의 기준 음성을 만듭니다. 모델과 voice는 013 노트북 설정(`gpt-4o-mini-tts`, `alloy`)을 따랐고, 영어가 되는지는 이 셀에서 처음 확인합니다. **실제 과금되므로** 결과를 `data/tts_cache/`에 저장해 두고 있으면 다시 호출하지 않습니다.

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

import tts   # tts.py: 문장 읽기, TTS 호출과 캐시, 호출 횟수(tts.calls)
from tts import tts_wav, CACHE_DIR

sentences = tts.load_sentences()
for s in sentences:
    print(s["id"], s["sentence"])

s1 = sentences[0]
ref_path = tts_wav(s1["sentence"], CACHE_DIR / f"s{int(s1['id']):02d}.wav")
print("기준 음성:", ref_path, f"({ref_path.stat().st_size // 1024} KB), API 호출 {tts.calls}회")
display(Audio(filename=str(ref_path)))

## S3. 내 녹음 인식 (faster-whisper)

`data/recordings/`의 `sNN_take1.*`(wav·m4a·mp3 모두 가능)를 읽어 Whisper `tiny`로 인식합니다(`language="en"`, `beam_size=1`). 노트북에서 직접 녹음하지 않고 미리 녹음한 파일을 씁니다. 이 셀의 출력에는 내 목소리의 인식 결과가 남으므로 커밋 전에 출력을 지웁니다.

In [ ]:
from stt import REC_DIR, find_recording, recognize   # stt.py로 옮긴 함수들

hyps = {}   # {문장 id: 인식 결과}
for s in sentences:
    path = find_recording(s["id"])
    if path is None:
        print(f"[{s['id']}] 녹음 없음: {REC_DIR}/s{int(s['id']):02d}_take1.*")
        continue
    hyps[s["id"]], sec = recognize(path)
    print(f"[{s['id']}] 정답: {s['sentence']}")
    print(f"    인식: {hyps[s['id']]}   ({path.suffix}, {sec:.1f}초)")

## S4. 채점과 틀린 부분 표시

정답 문장과 인식 결과를 `scoring`으로 비교합니다. 점수는 발음 점수가 아니라 **문장 인식 일치도**의 오류율입니다. 오류율은 1을 넘을 수 있어 "일치율"로 바꿔 표시하지 않고 치환(S)·삭제(D)·삽입(I) 개수를 함께 보여 줍니다. 틀린 부분은 `[정답→인식]`(치환), `[-정답-]`(삭제), `[+인식+]`(삽입)으로 표시합니다.

In [ ]:
for s in sentences:
    if s["id"] not in hyps:
        continue
    w = scoring.wer(s["sentence"], hyps[s["id"]])
    c = scoring.cer(s["sentence"], hyps[s["id"]])
    print(f"문장 #{s['id']}: \"{s['sentence']}\"")
    print(f"Whisper(tiny) 인식: \"{hyps[s['id']]}\"")
    print(f"WER {w['rate']:.3f}  (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / 정답 {w['N']}단어)")
    print(f"CER {c['rate']:.3f}  (치환 {c['S']}, 삭제 {c['D']}, 삽입 {c['I']} / 정답 {c['N']}글자)")
    print(scoring.format_diff(w["ops"]))
    print()

## (선택) 대조 실험: TTS 기준 음성을 같은 Whisper에 넣어 보기

3번 문장의 `we'll sing` 구간이 두 번 모두 틀렸습니다. 원인이 내 녹음·발음인지 `tiny` 모델의 한계인지 가르기 위해, TTS로 만든 3번 문장 기준 음성을 같은 설정으로 인식시킵니다. 표본이 1개(문장 1개, voice 1개, 1회)라서 **일반화하지 않고 이 문장에서의 참고 결과로만** 봅니다.

In [ ]:
s3 = sentences[2]
p3 = tts_wav(s3["sentence"], CACHE_DIR / f"s{int(s3['id']):02d}.wav")   # 이미 있으면 API를 부르지 않음
tts_hyp, sec = recognize(p3)
w = scoring.wer(s3["sentence"], tts_hyp)

print(f"정답          : {s3['sentence']}")
print(f"TTS 인식      : {tts_hyp}   ({sec:.1f}초, 이 커널의 API 호출 {tts.calls}회)")
print(f"내 녹음 인식  : {hyps[s3['id']]}")
print(f"TTS  WER {w['rate']:.3f} (치환 {w['S']}, 삭제 {w['D']}, 삽입 {w['I']} / {w['N']}단어)")
print(scoring.format_diff(w["ops"]))

## S4.5. 수업 내용 적용: 같은 문장의 TTS와 내 녹음을 소리 그림으로 비교

3번 문장(`We'll sing together until the morning comes.`)의 TTS 기준 음성과 내 녹음을 **같은 설정**으로 그립니다. 수업 코드를 가져와 썼습니다.

| 수업 내용 | 출처 | 이 그림에서의 역할 |
|---|---|---|
| WAV 읽기, 샘플링 레이트, 모노·16 kHz 변환 | 02, lesson04 4-D | 48 kHz(내 녹음)와 24 kHz(TTS)를 같은 16 kHz 모노로 맞춤 |
| STFT (창 512, 이동 128, hann) | 03 `compute_stft`, lesson04 | 시간별 주파수 분포(스펙트로그램) |
| Mel 필터 40개 + 로그 | lesson04 `logmel` | Log-Mel (Whisper가 모델 입력으로 쓰는 특징의 교육용 버전) |

**읽는 법**: 이 그림은 **참고 정보이며 평가가 아닙니다.** 스펙트로그램이 다르다고 해서 발음이 틀렸다는 뜻이 아닙니다. 두 소리의 길이·말 속도·무음 구간·에너지가 모이는 위치가 어떻게 다른지 눈으로 보는 용도입니다. 비교를 위해 두 파형은 최대 진폭을 같게 맞추고, 같은 색 범위로 그립니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import theme
theme.apply_style()   # 프로젝트 공통 색·글꼴 (theme.py, 초록 계열)

# 수업 코드를 모듈로 옮겼습니다(동작은 같음).
from audio import load_audio                  # audio.py: 어떤 녹음 파일이든 16 kHz 모노 float32 파형으로 (lesson04 4-D 순서)
from visualisation import logmel, plot_sound_pair, plot_error_spans
# visualisation.py: STFT 파워 → Mel 필터 → Log-Mel(dB) 계산(lesson04 실습 2-F·3-B)과 그림 그리기

In [ ]:
sid = sentences[2]["id"]
sources = [("TTS 기준 음성", CACHE_DIR / f"s{int(sid):02d}.wav"), ("내 녹음", find_recording(sid))]

sounds = []
for name, p in sources:
    x, sr0, ch = load_audio(p)
    x = x / max(np.abs(x).max(), 1e-9) * 0.9          # 최대 진폭을 맞춰 모양만 비교
    f = logmel(x, 16000)
    sounds.append((name, x))
    print(f"{name}: 원본 {sr0} Hz / {ch}채널 → 16000 Hz 모노, {len(x) / 16000:.2f}초, "
          f"STFT {f['power'].shape}, Log-Mel {f['log_mel'].shape}  (주파수·Mel칸, 시간)")

# 그리기는 visualisation.plot_sound_pair가 합니다(두 소리를 같은 설정·같은 색 범위로).
plot_sound_pair(sounds, f'문장 #{sid}: "{sentences[2]["sentence"]}"  (참고 정보, 평가 아님; STFT 창 512·이동 128, Mel 40, 같은 색 범위)')
plt.show()

### S4.5 결과 검토 (2026-11-01, 그림을 보고 쓴 관찰)

표시: **[확인]** 그림에서 바로 보이는 것, **[추정]** 해석, **[미확인]** 이 그림만으로는 알 수 없는 것. 이 그림은 참고 정보이며 평가가 아닙니다.

| 관찰 | 근거 | 해석 |
|---|---|---|
| 말하는 구간의 길이는 비슷합니다. TTS는 약 0.25~3.2초(약 3초), 내 녹음은 약 1.3~4.5초(약 3.2초)입니다. | **[확인]** 파형·스펙트로그램의 소리 구간 | 말 속도가 인식 실패의 주된 원인은 아닌 것 같습니다 **[추정]**(눈대중이며 단어별 시간은 재지 않았습니다). |
| 내 녹음은 앞의 약 1.3초가 거의 무음이고, 0.8초 근처에 짧고 넓은 대역의 소리가 있습니다. | **[확인]** | 숨소리·입소리·녹음 시작 소음일 수 있습니다 **[추정]**. Whisper는 무음 뒤 첫 단어를 놓치기 쉬운지는 **[미확인]**입니다. |
| TTS의 무음은 완전히 비어 있고(가장 어두운 색), 내 녹음의 무음에는 낮은 주파수(약 0.3 kHz 이하)에 가는 띠가 남아 있습니다. | **[확인]** 스펙트로그램 아래쪽 | 배경 소음이나 마이크의 저주파 성분일 수 있습니다 **[추정]**. 계획서의 `quality.py` "80 Hz 이하 에너지 비율" 참고 항목이 이런 것을 보려는 것입니다. |
| 모음 구간에 가로 줄무늬(고조파)가 두 소리 모두에 보입니다. TTS는 줄이 곧고 굵으며, 내 녹음은 줄이 위아래로 더 많이 휘어집니다. | **[확인]** 03 STFT 수업의 줄무늬와 같은 현상 | 줄이 휘어진 정도는 음높이(억양) 변화로 볼 수 있습니다 **[일반 지식]**. 내 억양이 TTS보다 더 움직였다는 뜻인지는 **[미확인]**입니다. |
| TTS는 단어 사이에 세로 빈틈이 뚜렷하고, 내 녹음은 1.3~2.0초 구간이 이어져 빈틈이 적습니다. | **[확인]** 세로 빈틈의 선명도 | 단어가 붙어서 읽혔을 가능성이 있습니다 **[추정]**. 이 구간이 `we'll sing`인지는 단어별 시간을 재지 않아 **[미확인]**입니다. |
| 내 녹음은 말하는 동안 4~8 kHz에도 에너지가 넓게 퍼져 있고, TTS는 특정 순간(자음으로 보이는 곳)에만 높습니다. | **[확인]**(두 소리의 최대 진폭을 같게 맞춘 뒤 비교) | 마이크·녹음 환경의 차이일 수 있습니다 **[추정]**. |

**정리**: 두 소리는 눈에 띄게 다르지만, 이 그림만으로는 "인식이 틀린 이유"를 하나로 정할 수 없습니다. 다음에 가장 싸게 확인할 수 있는 것은 두 가지입니다. ① `model.transcribe(..., word_timestamps=True)`로 단어별 시간을 얻어 그림 위에 겹쳐 그리기(어느 구간이 `we'll sing`인지 확인), ② 같은 문장을 앞 무음을 줄이고 다시 녹음해 비교하기.

**색 비교 메모**: 이 그림은 처음에 수업의 `magma`(스펙트로그램)와 `Greens_r`(Log-Mel)를 섞어 썼습니다. 지금은 프로젝트 공통 초록 램프(`theme.CMAP_SPEC`)로 통일했고, 근거는 검토 문서 8.26절에 있습니다.

## S4.6. 틀린 구간을 소리 그림 위에 표시하기

S4.5의 그림은 문장 전체를 한 장으로 그린 것이라 **어느 단어가 그림의 어디인지** 알 수 없었습니다. 이번에는 채점 결과와 시간을 연결합니다.

1. 내 녹음과 TTS를 각각 `word_timestamps=True`로 인식해 **단어별 시작·끝 시간**을 얻습니다(`stt.recognize_words`, 앞 무음이 첫 단어에 섞이는 것을 줄이려고 `vad_filter=True`).
2. `scoring`의 결과에서 **연속된 틀린 칸을 한 덩어리**로 묶습니다(`align.error_blocks`). 단어 수가 달라도(`we'll` ↔ `we are`) 한 덩어리로 비교할 수 있습니다.
3. 덩어리마다 정답 단어는 **TTS의 구간**으로, 인식된 단어는 **내 녹음의 구간**으로 바꿔(`align.locate_errors`) 두 그림에 같은 번호 `[k]`로 표시합니다. 한쪽에 단어가 없으면(삭제·삽입) 이웃 단어 사이의 점선 한 줄로 표시합니다.

**읽는 법**: 이 표시는 "내가 말한 소리 중 잘못 인식된 곳이 어디이고, 같은 위치의 기준 소리는 어떻게 생겼는지"를 나란히 보는 용도입니다. **원인을 알려 주지는 않습니다.** 단어 시간은 Whisper가 모델 내부 정보로 **추정한 값**이라 정확한 강제 정렬이 아니며(`tiny`는 오차가 클 수 있음), 구간은 대략의 위치입니다.

In [ ]:
import practice as flow   # practice.py: 연습 한 번의 순서(문장 → TTS → 점검 → 인식 → 채점 → 틀린 구간). 노트북 함수는 run_practice()라 이름이 겹치지 않지만, 모듈 이름과 구분하려고 flow로 부릅니다.
import report             # report.py: 결과 dict → 읽을 글

def show_error_spans(target, take=1):
    """틀린 덩어리의 시간 구간을 TTS(왼쪽)와 내 녹음(오른쪽)의 파형·스펙트로그램·Log-Mel 위에 같은 번호로 표시한다.
    인식·구간 계산은 flow.locate_errors, 글은 report.error_blocks, 그림은 visualisation.plot_error_spans가 맡는다.
    구간은 Whisper의 단어별 시간(추정값)이라 대략의 위치이고, 평가가 아니라 참고 정보다."""
    prep = flow.prepare(target, sentences)
    rec = find_recording(prep["id"], take) if prep["id"].isdigit() else None
    if rec is None:
        print("따라 읽은 녹음이 없습니다."); return
    out = flow.locate_errors(prep, {"recording": rec})      # 인식이 2번 돌아 몇 초 걸린다
    print(report.error_blocks(prep, out["loc"]))
    plot_error_spans(out["sounds"], out["loc"]["blocks"],
                     f'문장 #{prep["id"]}: 같은 번호 [k]는 같은 덩어리 (참고 정보, 평가 아님; 구간은 Whisper 단어 시간의 추정값)')
    plt.show()

In [ ]:
# 3번 문장: 내 녹음에서 잘못 인식된 덩어리와, 같은 위치의 TTS 소리를 나란히 봅니다 (TTS는 캐시, API 호출 0회).
# 위쪽 표(구간 숫자)와 그림의 [1] 표시가 같은 곳을 가리키는지 확인하세요. 새 인식이 2번 돌아 몇 초 걸립니다.
show_error_spans(3)

## S6. 연습 흐름: 듣고 → 따라 읽기 또는 받아쓰기 → 채점

한 문장을 위에서 아래로 한 번에 연습하는 화면입니다. 지금까지 만든 부분(TTS 캐시, 녹음 점검, Whisper 인식, `scoring`)을 `practice()` 하나로 이어 붙였습니다. 이 셀은 보여 주는 일만 하고, **순서는 `practice.py`, 글은 `report.py`** 가 맡습니다(같은 함수를 나중에 웹 화면에서도 부를 수 있도록 모듈로 분리).

| 모드 | 순서 |
|---|---|
| `"read"` 따라 읽기 | 문장과 뜻 확인 → 기준 음성 듣기 → **녹음 상태 점검** → (미리 녹음한 파일을) 인식 → 오류율과 틀린 부분 |
| `"dictation"` 받아쓰기 | 기준 음성 듣기(문장은 숨김) → 들은 대로 입력 → 오류율과 틀린 부분 → 정답과 뜻 공개 |

- 문장은 `sentences.csv`의 번호(`1`, `2`, `3`)로 부르거나, 직접 쓴 영어 문장을 문자열로 넣을 수 있습니다.
- **직접 입력한 문장은 TTS API로 전송되고 과금됩니다**(한 번 만든 음성은 `data/tts_cache/`에 저장해 다시 쓰지 않습니다). **직접 쓴 문장만 넣고, 상업 노래의 가사는 넣지 않습니다.**
- 점수는 발음 점수가 아니라 문장 인식 일치도의 오류율입니다. 받아쓰기는 STT를 거치지 않아 인식 오류가 섞이지 않고, 맞춤법·띄어쓰기 같은 내 쓰기 실수만 점수에 반영됩니다.
- 녹음 상태 점검의 경고는 안내일 뿐이며, 경고가 있어도 인식과 채점은 그대로 진행합니다.
- 받아쓰기의 `input()` 창은 VS Code에서 화면 위쪽에 나타납니다.

In [ ]:
def run_practice(target, mode="read", recording=None):
    """한 문장 연습. target: sentences.csv의 id 또는 직접 쓴 문장.
    mode="read": 문장 확인 → 듣기 → 녹음 점검 → 인식 → 채점 (recording을 안 주면 data/recordings/sNN_take1.*)
    mode="dictation": 듣기(문장은 숨김) → 들은 대로 입력 → 채점 → 정답 공개
    이 셀은 화면에 보여 주는 일만 한다. 순서는 practice.py(flow), 글은 report.py가 만든다."""
    if mode not in ("read", "dictation"):
        raise ValueError('mode는 "read" 또는 "dictation"입니다.')
    prep = flow.prepare(target, sentences)            # 문장 찾기 + 기준 음성(TTS, 캐시)
    if prep["error"]:
        print(prep["error"])
        return
    print(report.header(prep, mode))
    print(report.tts_status(prep))
    display(Audio(filename=str(prep["tts_path"])))

    if mode == "read":
        res = flow.read(prep, recording)               # 녹음 점검 → 인식 → 채점
    else:
        res = flow.dictation(prep, input("들은 대로 적어 보세요 > "))
    print(report.result(prep, res))

In [ ]:
# 1) 따라 읽기: 3번 문장을 듣고, 미리 녹음한 data/recordings/s03_take1.wav를 인식해 채점합니다 (TTS는 캐시, 호출 0회).
run_practice(3, "read")

In [ ]:
# 2) 받아쓰기: 1번 문장을 듣고 들은 대로 입력합니다 (입력창은 화면 위쪽에 뜹니다. TTS는 캐시, 호출 0회).
run_practice(1, "dictation")

In [ ]:
# 3) 직접 쓴 문장으로 받아쓰기: 문장을 바꿔 쓸 수 있습니다. 처음 쓰는 문장은 TTS를 새로 만들어 과금(1회)됩니다.
#    직접 쓴 문장만 넣고, 상업 노래의 가사는 넣지 않습니다.
run_practice("Where the river meets the sea.", "dictation")

In [ ]:
# 2026.11.1(일) 21h30 __ 테스트
import quality
print(quality.format_report(quality.check_file("data/recordings/s01_take1.wav")))

In [ ]:
import quality, glob
print(f"{'파일':14s}{'길이':>6s}{'RMS':>8s}{'최대':>6s}{'클리핑':>7s}{'무음':>6s}  경고")
for p in sorted(glob.glob("data/quality_test/*.wav")):
    r = quality.check_file(p); m = r["metrics"]
    print(f"{p.split('/')[-1]:14s}{m['seconds']:5.1f}s{m['rms_dbfs']:7.1f}{m['peak']:6.2f}{m['clip_ratio']*100:6.2f}%{m['silent_ratio']*100:5.0f}%  "
        + (" / ".join(w[:9] for w in r["warnings"]) or "없음"))